In [1]:
"""
Phase 3.5 (v2) — Model Improvement Sprint with XGBoost

Three improvements:
  1. Scale dataset: 1000 → 5000 students
  2. GridSearchCV: tune both Random Forest AND XGBoost
  3. Threshold tuning: on the winning classifier

Head-to-head comparison:
  RandomForest vs XGBoost for both regression and classification
  Best model per task is selected automatically (val metric)
  Only the winner gets evaluated on the test set
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve,
    precision_score, recall_score
)

from xgboost import XGBRegressor, XGBClassifier

In [2]:
print("=" * 70)
print("PHASE 3.5 (v2) — RF vs XGBoost, GridSearch, Threshold Tuning")
print("=" * 70)

# STEP 1: Regenerate Dataset at 5000 Students
print("\n[1/4] Scaling dataset to 5000 students...")

np.random.seed(42)
N = 5000

study_hours             = np.clip(np.random.normal(15, 5, N), 2, 40)
avg_session_length      = np.clip(np.random.normal(60, 20, N), 15, 180)
study_sessions_per_week = np.clip((study_hours * 60) / avg_session_length, 2, 25)
focus_ratio             = np.clip(np.random.beta(5, 2, N), 0.3, 0.98)
late_night_ratio        = np.clip(np.random.beta(2, 5, N), 0.0, 0.8)
days_until_deadline     = np.clip(np.random.gamma(3, 2, N), 0.5, 14)
video_completion_rate   = np.clip(np.random.beta(4, 2, N), 0.2, 1.0)
practice_attempts       = np.clip(np.random.gamma(2, 8, N), 0, 50)
forum_participation     = np.clip(np.random.poisson(2, N), 0, 15)
study_streak            = np.clip(np.random.gamma(3, 4, N), 1, 30)
ability_factor          = np.random.normal(0, 8, N)

final_grade = np.full(N, 20.0)
final_grade += study_hours           * 1.5
final_grade += focus_ratio           * 16
final_grade += avg_session_length    / 60 * 2
final_grade += video_completion_rate * 8
final_grade += np.sqrt(practice_attempts) * 2
final_grade += np.log1p(forum_participation) * 2
final_grade += (study_streak / 30)   * 7
final_grade += days_until_deadline   * 0.4
final_grade -= late_night_ratio      * 14
final_grade -= (study_sessions_per_week > 20) * 6
final_grade += ability_factor
final_grade += np.random.normal(0, 5, N)
final_grade  = np.round(np.clip(final_grade, 0, 100), 1)

PASS_THRESHOLD = 60
pass_fail = (final_grade >= PASS_THRESHOLD).astype(int)

print(f"{N} students generated")
print(f"Pass rate: {pass_fail.mean()*100:.1f}%  "
      f"| Fail: {(pass_fail==0).sum()}  Pass: {(pass_fail==1).sum()}")

FEATURE_COLS = [
    'total_study_hours', 'avg_session_length', 'study_sessions_per_week',
    'focus_ratio', 'late_night_study_ratio', 'days_until_deadline_avg',
    'video_completion_rate', 'practice_problem_attempts',
    'forum_participation', 'study_streak_days'
]

data = pd.DataFrame({
    'total_study_hours':         np.round(study_hours, 1),
    'avg_session_length':        np.round(avg_session_length, 1),
    'study_sessions_per_week':   np.round(study_sessions_per_week, 1),
    'focus_ratio':               np.round(focus_ratio, 3),
    'late_night_study_ratio':    np.round(late_night_ratio, 3),
    'days_until_deadline_avg':   np.round(days_until_deadline, 1),
    'video_completion_rate':     np.round(video_completion_rate, 3),
    'practice_problem_attempts': np.round(practice_attempts, 0),
    'forum_participation':       forum_participation,
    'study_streak_days':         np.round(study_streak, 0),
    'final_grade':               final_grade,
    'pass_fail':                 pass_fail
})

#Preprocessing pipeline
X       = data[FEATURE_COLS].copy()
y_grade = data['final_grade'].copy()
y_pass  = data['pass_fail'].copy()

X_train, X_temp, yg_train, yg_temp, yp_train, yp_temp = train_test_split(
    X, y_grade, y_pass, test_size=0.30, random_state=42, stratify=y_pass
)
X_val, X_test, yg_val, yg_test, yp_val, yp_test = train_test_split(
    X_temp, yg_temp, yp_temp, test_size=0.50, random_state=42, stratify=yp_temp
)

scaler = StandardScaler()
scaler.fit(X_train)

def preprocess(df):
    scaled = pd.DataFrame(scaler.transform(df), columns=FEATURE_COLS, index=df.index)
    #Feature engineering
    scaled['study_efficiency'] = scaled['total_study_hours'] * scaled['focus_ratio']
    scaled['procrastination_penalty'] = (
        scaled['late_night_study_ratio'] / (scaled['days_until_deadline_avg'] + 0.1)
    )
    return scaled

X_train_e = preprocess(X_train)
X_val_e   = preprocess(X_val)
X_test_e  = preprocess(X_test)

ALL_FEATURES = list(X_train_e.columns)  # 12 features

print(f"\nTrain: {len(X_train_e)} | Val: {len(X_val_e)} | Test: {len(X_test_e)}")
print(f" Features: {ALL_FEATURES}")

PHASE 3.5 (v2) — RF vs XGBoost, GridSearch, Threshold Tuning

[1/4] Scaling dataset to 5000 students...
5000 students generated
Pass rate: 80.1%  | Fail: 993  Pass: 4007

Train: 3500 | Val: 750 | Test: 750
 Features: ['total_study_hours', 'avg_session_length', 'study_sessions_per_week', 'focus_ratio', 'late_night_study_ratio', 'days_until_deadline_avg', 'video_completion_rate', 'practice_problem_attempts', 'forum_participation', 'study_streak_days', 'study_efficiency', 'procrastination_penalty']


In [3]:
# STEP 2: GridSearchCV — RF vs XGBoost (Regression)
print("\n[2/4] GridSearchCV — Regression (RF vs XGBoost)")
print("-" * 50)

#Random Forest Regressor grid
rf_reg_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        'n_estimators':     [100, 200, 300],
        'max_depth':        [6, 8, 12, None],
        'min_samples_leaf': [3, 5, 10],
    },
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=0
)
rf_reg_grid.fit(X_train_e, yg_train)
rf_reg_best  = rf_reg_grid.best_estimator_
rf_reg_cv    = -rf_reg_grid.best_score_
rf_val_pred  = rf_reg_best.predict(X_val_e)
rf_reg_val_mae = mean_absolute_error(yg_val, rf_val_pred)
rf_reg_val_r2  = r2_score(yg_val, rf_val_pred)

print(f"\n  RandomForest Regressor")
print(f"    Best params: {rf_reg_grid.best_params_}")
print(f"    CV MAE:      {rf_reg_cv:.2f} pts")
print(f"    Val MAE:     {rf_reg_val_mae:.2f} pts  |  Val R²: {rf_reg_val_r2:.3f}")

#XGBoost Regressor grid
# XGBoost has its own key hyperparameters:
#   learning_rate: step size per tree (smaller = more trees needed but better generalisation)
#   subsample:     fraction of training rows per tree (reduces overfitting)
#   colsample_bytree: fraction of features per tree (like RF's max_features)
xgb_reg_grid = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {
        'n_estimators':      [100, 200, 300],
        'max_depth':         [3, 4, 6],
        'learning_rate':     [0.01, 0.05, 0.1],
        'subsample':         [0.7, 0.8, 1.0],
        'colsample_bytree':  [0.7, 0.8, 1.0],
    },
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=0
)
xgb_reg_grid.fit(X_train_e, yg_train)
xgb_reg_best   = xgb_reg_grid.best_estimator_
xgb_reg_cv     = -xgb_reg_grid.best_score_
xgb_val_pred   = xgb_reg_best.predict(X_val_e)
xgb_reg_val_mae = mean_absolute_error(yg_val, xgb_val_pred)
xgb_reg_val_r2  = r2_score(yg_val, xgb_val_pred)

print(f"\n  XGBoost Regressor")
print(f"    Best params: {xgb_reg_grid.best_params_}")
print(f"    CV MAE:      {xgb_reg_cv:.2f} pts")
print(f"    Val MAE:     {xgb_reg_val_mae:.2f} pts  |  Val R²: {xgb_reg_val_r2:.3f}")

# Pick winner (lower val MAE)
if rf_reg_val_mae <= xgb_reg_val_mae:
    best_reg       = rf_reg_best
    best_reg_name  = "RandomForest"
    best_reg_mae   = rf_reg_val_mae
    best_reg_r2    = rf_reg_val_r2
else:
    best_reg       = xgb_reg_best
    best_reg_name  = "XGBoost"
    best_reg_mae   = xgb_reg_val_mae
    best_reg_r2    = xgb_reg_val_r2

print(f"\nRegression winner: {best_reg_name}  (Val MAE={best_reg_mae:.2f})")

# STEP 3: GridSearchCV — RF vs XGBoost (Classification)
print("\n[3/4] GridSearchCV — Classification (RF vs XGBoost)")
print("-" * 50)

#Random Forest Classifier grid
rf_clf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    {
        'n_estimators':     [100, 200, 300],
        'max_depth':        [6, 8, 12, None],
        'min_samples_leaf': [3, 5, 10],
    },
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
rf_clf_grid.fit(X_train_e, yp_train)
rf_clf_best   = rf_clf_grid.best_estimator_
rf_clf_cv_auc = rf_clf_grid.best_score_
rf_clf_proba  = rf_clf_best.predict_proba(X_val_e)[:, 1]
rf_clf_val_auc = roc_auc_score(yp_val, rf_clf_proba)
rf_clf_val_f1f = f1_score(yp_val, (rf_clf_proba >= 0.5).astype(int),
                          pos_label=0, average='binary')

print(f"\n  RandomForest Classifier")
print(f"    Best params: {rf_clf_grid.best_params_}")
print(f"    CV AUC:      {rf_clf_cv_auc:.3f}")
print(f"    Val AUC:     {rf_clf_val_auc:.3f}  |  Val F1-Fail: {rf_clf_val_f1f:.3f}")

# XGBoost Classifier grid
# scale_pos_weight: upweights the minority class during training
# equivalent to class_weight='balanced' in sklearn
neg   = int((yp_train == 0).sum())
pos   = int((yp_train == 1).sum())
scale = neg / pos   # ~0.25: fail class gets 4x the penalty

xgb_clf_grid = GridSearchCV(
    XGBClassifier(
        scale_pos_weight=scale,
        random_state=42,
        verbosity=0,
        eval_metric='logloss',
        use_label_encoder=False
    ),
    {
        'n_estimators':     [100, 200, 300],
        'max_depth':        [3, 4, 6],
        'learning_rate':    [0.01, 0.05, 0.1],
        'subsample':        [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0],
    },
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
xgb_clf_grid.fit(X_train_e, yp_train)
xgb_clf_best    = xgb_clf_grid.best_estimator_
xgb_clf_cv_auc  = xgb_clf_grid.best_score_
xgb_clf_proba   = xgb_clf_best.predict_proba(X_val_e)[:, 1]
xgb_clf_val_auc = roc_auc_score(yp_val, xgb_clf_proba)
xgb_clf_val_f1f = f1_score(yp_val, (xgb_clf_proba >= 0.5).astype(int),
                           pos_label=0, average='binary')

print(f"\n  XGBoost Classifier")
print(f"    Best params: {xgb_clf_grid.best_params_}")
print(f"    CV AUC:      {xgb_clf_cv_auc:.3f}")
print(f"    Val AUC:     {xgb_clf_val_auc:.3f}  |  Val F1-Fail: {xgb_clf_val_f1f:.3f}")

# --- Pick winner (higher val AUC) ---
if rf_clf_val_auc >= xgb_clf_val_auc:
    best_clf        = rf_clf_best
    best_clf_name   = "RandomForest"
    best_clf_proba  = rf_clf_proba
    best_clf_val_auc = rf_clf_val_auc
else:
    best_clf        = xgb_clf_best
    best_clf_name   = "XGBoost"
    best_clf_proba  = xgb_clf_proba
    best_clf_val_auc = xgb_clf_val_auc

print(f"\nClassification winner: {best_clf_name}  (Val AUC={best_clf_val_auc:.3f})")


[2/4] GridSearchCV — Regression (RF vs XGBoost)
--------------------------------------------------

  RandomForest Regressor
    Best params: {'max_depth': 12, 'min_samples_leaf': 3, 'n_estimators': 200}
    CV MAE:      7.90 pts
    Val MAE:     7.83 pts  |  Val R²: 0.403

  XGBoost Regressor
    Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.7}
    CV MAE:      7.72 pts
    Val MAE:     7.67 pts  |  Val R²: 0.435

Regression winner: XGBoost  (Val MAE=7.67)

[3/4] GridSearchCV — Classification (RF vs XGBoost)
--------------------------------------------------

  RandomForest Classifier
    Best params: {'max_depth': 12, 'min_samples_leaf': 10, 'n_estimators': 200}
    CV AUC:      0.805
    Val AUC:     0.808  |  Val F1-Fail: 0.527

  XGBoost Classifier
    Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
    CV AUC:      0.810
    Val AUC:     0.813 

In [37]:
# STEP 4: Threshold Tuning on Winning Classifier
print("\n[4/4] Threshold tuning on validation set...")
print("-" * 50)

threshold_log  = []
best_threshold = 0.5
best_f1_fail   = f1_score(yp_val, (best_clf_proba >= 0.5).astype(int),
                          pos_label=0, average='binary')


for thresh in np.arange(0.10, 0.90, 0.01):
    preds = (best_clf_proba >= thresh).astype(int)
    if len(np.unique(preds)) < 2:
        continue
    f1f  = f1_score(yp_val, preds, pos_label=0, average='binary')
    f1m  = f1_score(yp_val, preds, average='macro')
    acc  = accuracy_score(yp_val, preds)
    rec  = ((preds==0) & (yp_val==0)).sum() / (yp_val==0).sum()
    threshold_log.append({'threshold': thresh, 'f1_fail': f1f,
                          'f1_macro': f1m, 'accuracy': acc, 'recall_fail': rec})
    if f1f > best_f1_fail:
        best_f1_fail   = f1f
        best_threshold = thresh

thresh_df = pd.DataFrame(threshold_log)
#best_threshold = 0.60 #expirement

print(f"\n  Default threshold (0.50) F1-Fail: "
      f"{f1_score(yp_val, (best_clf_proba>=0.5).astype(int), pos_label=0, average='binary'):.3f}")
print(f"  Optimal threshold ({best_threshold:.2f}) F1-Fail: {best_f1_fail:.3f}")

print(f"\n  Threshold sweep (val set):")
print(f"  {'Thresh':>7} | {'F1-Fail':>8} | {'Recall-Fail':>12} | {'Accuracy':>9}")
print(f"  {'-'*47}")
for t in [0.30, 0.40, best_threshold, 0.50, 0.60, 0.70]:
    row = thresh_df[thresh_df['threshold'].between(t-0.006, t+0.006)]
    if len(row) == 0:
        continue
    r   = row.iloc[0]
    marker = " ← optimal" if abs(t - best_threshold) < 0.015 else ""
    print(f"  {t:>7.2f} | {r['f1_fail']:>8.3f} | {r['recall_fail']:>11.1%} | "
          f"{r['accuracy']:>8.1%}{marker}")


[4/4] Threshold tuning on validation set...
--------------------------------------------------

  Default threshold (0.50) F1-Fail: 0.519
  Optimal threshold (0.44) F1-Fail: 0.541

  Threshold sweep (val set):
   Thresh |  F1-Fail |  Recall-Fail |  Accuracy
  -----------------------------------------------
     0.30 |    0.520 |       47.0% |    82.8%
     0.40 |    0.523 |       57.7% |    79.1%
     0.44 |    0.541 |       63.8% |    78.5% ← optimal
     0.50 |    0.519 |       67.1% |    75.3%
     0.60 |    0.504 |       76.5% |    70.1%
     0.70 |    0.472 |       83.9% |    62.7%


In [38]:
#RETRAIN BEST MODELS ON TRAIN + VALIDATION

print("\nRetraining winning models on Train + Validation set...")

# Combine train and validation sets
X_trainval_e = pd.concat([X_train_e, X_val_e])
yg_trainval  = pd.concat([yg_train, yg_val])
yp_trainval  = pd.concat([yp_train, yp_val])

# Retrain Regressor
if best_reg_name == "RandomForest":
    best_reg = RandomForestRegressor(**rf_reg_grid.best_params_, random_state=42)
else:
    best_reg = XGBRegressor(**xgb_reg_grid.best_params_, random_state=42, verbosity=0)

best_reg.fit(X_trainval_e, yg_trainval)

# Retrain Classifier
if best_clf_name == "RandomForest":
    best_clf = RandomForestClassifier(
        **rf_clf_grid.best_params_,
        class_weight='balanced',
        random_state=42
    )
else:
    best_clf = XGBClassifier(
        **xgb_clf_grid.best_params_,
        scale_pos_weight=scale,
        random_state=42,
        verbosity=0,
        eval_metric='logloss',
        use_label_encoder=False
    )

best_clf.fit(X_trainval_e, yp_trainval)

print("Retraining complete.")


Retraining winning models on Train + Validation set...
Retraining complete.


In [39]:
# FINAL TEST SET EVALUATION
print("\n" + "=" * 70)
print("FINAL TEST EVALUATION (test set touched once)")
print("=" * 70)

# Regression
test_grade_pred  = best_reg.predict(X_test_e)
test_mae   = mean_absolute_error(yg_test, test_grade_pred)
test_rmse  = np.sqrt(mean_squared_error(yg_test, test_grade_pred))
test_r2    = r2_score(yg_test, test_grade_pred)

# Cross-val on train for confidence interval
cv_scores = cross_val_score(best_reg, X_train_e, yg_train,
                            cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
cv_mae = -cv_scores

# Classification
test_pass_proba  = best_clf.predict_proba(X_test_e)[:, 1]
test_pass_pred   = (test_pass_proba >= best_threshold).astype(int)
test_acc     = accuracy_score(yp_test, test_pass_pred)
test_f1      = f1_score(yp_test, test_pass_pred, average='macro')
test_f1_fail = f1_score(yp_test, test_pass_pred, pos_label=0, average='binary')
test_auc     = roc_auc_score(yp_test, test_pass_proba)
test_precision_fail = precision_score(
    yp_test, test_pass_pred, pos_label=0, average='binary'
)
test_recall_fail = recall_score(
    yp_test, test_pass_pred, pos_label=0, average='binary'
)

print(f"""
  REGRESSION  → {best_reg_name} (tuned, 5k students)
    Test MAE:   {test_mae:.2f} pts
    Test RMSE:  {test_rmse:.2f} pts
    Test R²:    {test_r2:.3f}
    CV MAE:     {cv_mae.mean():.2f} ± {cv_mae.std():.2f} pts  (5-fold, train set)

  CLASSIFICATION  → {best_clf_name} (tuned, threshold={best_threshold:.2f}, 5k students)
    Test Accuracy:      {test_acc*100:.1f}%
    Test F1 Macro:      {test_f1:.3f}
    Test F1-Fail:       {test_f1_fail:.3f}
    Precision (Fail):   {test_precision_fail:.3f}
    Recall (Fail):      {test_recall_fail:.3f}
    Test ROC-AUC:       {test_auc:.3f}
""")


print("  Success Criteria:")
print(f"  {'✅✅' if test_mae < 10 else '❗'}  Grade MAE < 10 pts           (got {test_mae:.2f})")
print(f"  {'✅' if test_auc >= 0.80 else '❌'}  ROC-AUC ≥ 0.80             (got {test_auc:.3f})")
print(f"  {'✅' if test_f1_fail >= 0.55 else '❌'}  F1-Fail ≥ 0.55           (got {test_f1_fail:.3f})")
print(f"  {'✅' if test_precision_fail >= 0.60 else '❌'}  Precision-Fail ≥ 0.60    (got {test_precision_fail:.3f})")
print(f"  {'✅' if test_recall_fail >= 0.60 else '❌'}  Recall-Fail ≥ 0.60      (got {test_recall_fail:.3f})")


print(f"\n  Classification Report (Test):")
report = classification_report(yp_test, test_pass_pred, target_names=['Fail','Pass'])
for line in report.split('\n'):
    print(f"    {line}")


FINAL TEST EVALUATION (test set touched once)

  REGRESSION  → XGBoost (tuned, 5k students)
    Test MAE:   7.92 pts
    Test RMSE:  9.88 pts
    Test R²:    0.395
    CV MAE:     7.72 ± 0.24 pts  (5-fold, train set)

  CLASSIFICATION  → XGBoost (tuned, threshold=0.44, 5k students)
    Test Accuracy:      77.9%
    Test F1 Macro:      0.703
    Test F1-Fail:       0.554
    Precision (Fail):   0.462
    Recall (Fail):      0.691
    Test ROC-AUC:       0.812

  Success Criteria:
  ✅✅  Grade MAE < 10 pts           (got 7.92)
  ✅  ROC-AUC ≥ 0.80             (got 0.812)
  ✅  F1-Fail ≥ 0.55           (got 0.554)
  ❌  Precision-Fail ≥ 0.60    (got 0.462)
  ✅  Recall-Fail ≥ 0.60      (got 0.691)

  Classification Report (Test):
                  precision    recall  f1-score   support
    
            Fail       0.46      0.69      0.55       149
            Pass       0.91      0.80      0.85       601
    
        accuracy                           0.78       750
       macro avg       0.

In [40]:
# HEAD-TO-HEAD SUMMARY TABLE
print("\n" + "=" * 70)
print("HEAD-TO-HEAD SUMMARY (Validation Set)")
print("=" * 70)
print(f"\n {'Model':<30} {'Val MAE':>8} {'Val R²':>8}")
print(f" {'-'*48}")
print(f" {'RandomForest Regressor':<30} {rf_reg_val_mae:>8.2f} {rf_reg_val_r2:>8.3f}")
print(f" {'XGBoost Regressor':<30} {xgb_reg_val_mae:>8.2f} {xgb_reg_val_r2:>8.3f}")
print(f" {'WINNER → ' + best_reg_name:<30}")
print(f"\n {'Model':<30} {'Val AUC':>8} {'F1-Fail':>8}")
print(f" {'-'*48}")
print(f" {'RandomForest Classifier':<30} {rf_clf_val_auc:>8.3f} {rf_clf_val_f1f:>8.3f}")
print(f" {'XGBoost Classifier':<30} {xgb_clf_val_auc:>8.3f} {xgb_clf_val_f1f:>8.3f}")
print(f" {'WINNER → ' + best_clf_name:<30}")

# SAVE ARTIFACTS
preprocessed = {
    'X_train': X_train_e,
    'X_val': X_val_e,
    'X_test': X_test_e,
    'y_grade_train': yg_train,
    'y_grade_val': yg_val,
    'y_grade_test': yg_test,
    'y_pass_train': yp_train,
    'y_pass_val': yp_val,
    'y_pass_test': yp_test,
    'scaler': scaler,
    'feature_columns': ALL_FEATURES
}

with open('preprocessed_data.pkl', 'wb') as f:
    pickle.dump(preprocessed, f)

model_artifacts = {
    'regressor': best_reg,
    'classifier': best_clf,
    'regressor_name': best_reg_name,
    'classifier_name': best_clf_name,
    'best_threshold': best_threshold,
    'feature_columns': ALL_FEATURES,
    'scaler': scaler,
    'test_metrics': {
        'reg_mae': round(test_mae, 2),
        'reg_rmse': round(test_rmse, 2),
        'reg_r2': round(test_r2, 3),
        'clf_acc': round(test_acc, 3),
        'clf_f1': round(test_f1, 3),
        'clf_f1_fail': round(test_f1_fail, 3),
        'clf_auc': round(test_auc, 3),
        'threshold': round(best_threshold, 2),
        'clf_precision_fail': round(test_precision_fail, 3),
        'clf_recall_fail': round(test_recall_fail, 3),
    }
}

with open('trained_models.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("\nSaved: trained_models.pkl")


HEAD-TO-HEAD SUMMARY (Validation Set)

 Model                           Val MAE   Val R²
 ------------------------------------------------
 RandomForest Regressor             7.83    0.403
 XGBoost Regressor                  7.67    0.435
 WINNER → XGBoost              

 Model                           Val AUC  F1-Fail
 ------------------------------------------------
 RandomForest Classifier           0.808    0.527
 XGBoost Classifier                0.813    0.519
 WINNER → XGBoost              

Saved: trained_models.pkl
